## Kodövningar (Coding exercises)

Questions 8–12 below are worked with real, executed code. Datasets used (see the intro cell for exact sources): `data_01.csv`, `salary_dataset.csv`, `mpg.csv`, `housing.csv` — all placed in the same folder as this notebook.

## 8. Explain what the code below does. Why is it important to be able to save a model?

In [1]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from joblib import dump, load

X, y = make_regression(n_samples=20000, n_features=3, noise=0.1)
model = LinearRegression().fit(X, y)

dump(model, "linear_model.joblib")
model_loaded = load("linear_model.joblib")
print(model_loaded.predict(X[:5]))

[ 130.27134228 -117.31680357   76.37439041  -22.94370256   45.06485566]



**What the code does, line by line:**

1. `from sklearn.datasets import make_regression` / `from sklearn.linear_model import LinearRegression` / `from joblib import dump, load` — imports a synthetic-regression-data generator, the linear regression model class, and the two `joblib` functions used to save (`dump`) and load (`load`) Python objects to/from disk.
2. `X, y = make_regression(n_samples=20000, n_features=3, noise=0.1)` — generates a synthetic regression dataset with 20,000 observations and 3 independent variables (features), with a small amount of Gaussian noise added to `y` so the relationship isn't perfectly deterministic.
3. `model = LinearRegression().fit(X, y)` — instantiates a linear regression model and trains it on the generated data in a single line (equivalent to the book's two-step `LinearRegression()` then `.fit(X, y)` pattern from Sections 1.3.4 and 2.4.2).
4. `dump(model, "linear_model.joblib")` — this is the key step: it **saves the entire trained model object to disk**, as a file called `linear_model.joblib`, in the same folder as the script. This mirrors exactly the book's own demonstration in Section 2.1.7 (p. 61), which uses `joblib.dump(model, 'our_linear_model.pkl')` for the same purpose.
5. `model_loaded = load("linear_model.joblib")` — loads that saved file back from disk into a new variable, `model_loaded`. This model object is functionally identical to `model` — same learned parameters (intercept and coefficients) — but it did not need to be retrained; it was reconstructed directly from the saved file.
6. `print(model_loaded.predict(X[:5]))` — uses the *loaded* (not retrained) model to make predictions on the first 5 rows of `X`, and prints the predicted values.

**Why is it important to be able to save a model?** The book makes this point directly on p. 61: once we've trained a model we intend to reuse, we save it so that it doesn't need to be retrained every time we want to use it ("När vi tränat en modell som vi tänkt återanvända så sparas modellen så att den inte behöver tränas om varje gång vi ska använda den"). In this exercise's code, training took only a moment because the dataset and model are small, but in general, training can be slow — for models trained on large datasets, or more complex models, training can take minutes, hours, or even much longer. Saving a trained model means:

- You can **reuse it instantly** later — e.g. after restarting your computer, or in an entirely separate script/program — without repeating the (potentially costly) training step.
- It is what actually makes **putting a model into production** possible in practice (Section 2.1.7's own MLOps discussion, p. 62): a model that predicts, say, churn risk in a live application or writes predictions into a database needs to be loaded and used repeatedly, on demand, without retraining itself before every single prediction.
- The saved file can be **moved to a different machine** (e.g. a production server) that only needs to load and use the model — it doesn't need access to the original training data or the time/resources it took to train it.

*Source: Section 2.1.7, pp. 61–62, code example with `joblib.dump()` / `joblib.load()`.*

In [2]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from joblib import dump, load

X, y = make_regression(n_samples=20000, n_features=3, noise=0.1, random_state=42)
model = LinearRegression().fit(X, y)

dump(model, "linear_model.joblib")
model_loaded = load("linear_model.joblib")
print(model_loaded.predict(X[:5]))

[ 105.19825923 -124.46546207  -15.07418334  103.66450947   92.8938913 ]


*(A `random_state=42` was added above only to make the printed numbers reproducible on every rerun — the exercise's own code doesn't need this to demonstrate saving/loading.)*

## 9. This exercise consists of several steps as described below.

a) Read the dataset `data_01.csv` with the `read_csv()` function from Pandas. The function returns a DataFrame.
b) Split the dataset into X and y.
c) Split the data further into a training, a validation, and a test set with `train_test_split()`. Let 20% of the data be test data and 15% of the remaining data be validation data.
d) Train two arbitrary regression models (e.g. `LinearRegression` and `DecisionTreeRegressor`) on the training data.
e) Evaluate the models on the validation data.
f) Retrain the best-performing model on both the training and validation data.
g) Evaluate the model on the test data.
h) Retrain the model on the entire dataset.

`data_01.csv` (from the book's own exercise dataset folder) has 198 rows and 6 columns: five features `x1`–`x5` and a continuous target column `target`, so it is treated as a regression problem exactly as Chapter 1 defines it (Section 1.2, p. 16) — `target` is the dependent variable `y`.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

# a) Read data_01.csv
df = pd.read_csv("data_01.csv")
print(df.shape)
df.head()

(199, 6)


,x1,x2,x3,x4,x5,target
0,0.743487,1.072825,1.332911,-1.244771,0.344978,220.173943
1,0.835264,0.202184,0.966480,0.745883,-0.033773,175.873929
2,-1.103234,0.030615,-0.140385,0.727683,-2.831224,-162.270054
3,1.210186,1.685258,-0.394123,0.719024,-2.166585,165.930461
4,0.474577,0.647737,-0.451812,-0.409472,-0.051473,43.250511


In [4]:
# b) Split the dataset into X and y
X = df.drop(columns=["target"])
y = df["target"]

In [5]:
# c) Split further into train, validation and test sets:
#    20% test data, then 15% of the *remaining* data as validation data
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.15, random_state=42)

print("X_train:", X_train.shape, " X_val:", X_val.shape, " X_test:", X_test.shape)

X_train: (135, 5)  X_val: (24, 5)  X_test: (40, 5)


In [16]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# d) Train models
lin_reg = LinearRegression().fit(X_train, y_train)
tree_reg = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)

# e) Evaluate on validation data
rmse_lin_val = np.sqrt(mean_squared_error(y_val, lin_reg.predict(X_val)))
rmse_tree_val = np.sqrt(mean_squared_error(y_val, tree_reg.predict(X_val)))

print(f"RMSE Linear Regression (validation): {rmse_lin_val:.4f}")
print(f"RMSE Decision Tree (validation):     {rmse_tree_val:.4f}")

best_name = "Linear Regression" if rmse_lin_val < rmse_tree_val else "Decision Tree"
print("Best-performing model on validation data:", best_name)


RMSE Linear Regression (validation): 3.5923
RMSE Decision Tree (validation):     99.3594
Best-performing model on validation data: Linear Regression
